# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
from typing import Literal, Optional, cast
from chromadb.api.types import Embeddable, EmbeddingFunction
from pydantic import BaseModel, Field
from tavily import TavilyClient
from lib.agents import Agent, AgentState
from lib.tooling import tool
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.parsers import PydanticOutputParser
from lib.memory import LongTermMemory, MemoryFragment
from lib.state_machine import Run
from lib.vector_db import VectorStoreManager


In [3]:
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
assert os.getenv("CHROMA_DB_PATH"), "CHROMA_DB_PATH is not set"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# Reuse the vector DB built in Part 1: same path, collection name and embedding function
chroma_client = chromadb.PersistentClient(path=os.environ["CHROMA_DB_PATH"])
embedding_fn = cast(EmbeddingFunction[Embeddable], embedding_functions.OpenAIEmbeddingFunction())
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)


@tool
def retrieve_game(query: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    Results with Platform "Cached web search" are earlier web searches saved to the DB;
    their Sources field holds the URLs to cite.
    """
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["metadatas", "distances"],
    )
    metadatas = (results["metadatas"] or [[]])[0]
    distances = (results["distances"] or [[]])[0]

    # Show this step in the output so the retrieve -> evaluate -> search flow is visible
    print(f"[retrieve_game] query={query!r}")
    for metadata, distance in zip(metadatas, distances):
        details = ", ".join(str(v) for v in (metadata.get("Platform"), metadata.get("YearOfRelease")) if v is not None)
        print(f"  - {metadata.get('Name')} ({details}) distance={distance:.3f}")

    # Each game's metadata holds the full JSON record (Name, Platform, YearOfRelease, Description, ...)
    return [dict(metadata) for metadata in metadatas]

#### Evaluate Retrieval Tool

In [5]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    confidence: Literal["high", "medium", "low"] = Field(description=(
        "How confident you are that the documents answer the question: high = they state the answer "
        "directly, medium = they answer it only partly or indirectly, low = they don't answer it"
    ))
    description: str = Field(description="Detailed explanation of the evaluation result")


# LLM as judge: temperature 0 keeps the verdicts consistent between runs
judge_llm = LLM(model="gpt-4o-mini", temperature=0.0)


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict[str, str]]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: high, medium or low confidence that the documents answer the question
    - description: description about the evaluation result
    """
    messages = [
        SystemMessage(content=(
            "You are an expert evaluator of retrieval results for a video game research assistant. "
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Rate your confidence that they answer it as high, medium or low. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )),
        UserMessage(content=(
            f"Question: {question}\n\n"
            f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2)}"
        )),
    ]
    response = judge_llm.invoke(messages, response_format=EvaluationReport)
    report = PydanticOutputParser(model_class=EvaluationReport).parse(response)

    # Show this step in the output so the retrieve -> evaluate -> search flow is visible
    print(f"[evaluate_retrieval] useful={report.useful} confidence={report.confidence}: {report.description}")

    return report.model_dump()

#### Game Web Search Tool

In [6]:
import hashlib
from datetime import date

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

WEB_CACHE_TYPE = "web_search_cache"


def cache_web_results(question: str, answer: Optional[str], results: list[dict]) -> str:
    """Save a web search into the internal vector DB so retrieve_game finds it next time"""
    # Same question (ignoring case/spacing) -> same id, so repeats overwrite instead of duplicating
    normalized = " ".join(question.lower().split())
    doc_id = "web-" + hashlib.sha1(normalized.encode("utf-8")).hexdigest()[:16]

    snippets = "\n".join(f"- {r['title']}: {r['content']}" for r in results)
    sources = ", ".join(r["url"] for r in results if r.get("url"))
    description = f"{answer or ''}\n{snippets}".strip()[:4000]

    collection.upsert(
        ids=[doc_id],
        # The question is part of the embedded text so the same question matches it closely
        documents=[f"Web search for: {question}\n{description}"],
        metadatas=[{
            "Name": f"Web search: {question}",
            "Platform": "Cached web search",
            "Description": description,
            "Question": question,
            "Sources": sources,
            "CachedOn": date.today().isoformat(),
            "Type": WEB_CACHE_TYPE,
        }],
    )
    return doc_id


def clear_web_search_cache() -> None:
    """Remove every cached web search from the internal vector DB (the 15 games stay)"""
    collection.delete(where={"Type": WEB_CACHE_TYPE})


@tool
def game_web_search(question: str) -> dict:
    """
    Web search: Searches the internet for game industry information that the internal
    vector DB doesn't have (e.g. games outside the dataset, recent releases, studio news).
    Results are saved to the internal vector DB, so the same question won't need a web search again.
    args:
    - question: a question about game industry.

    You'll receive a short answer plus a list of results, each with title, url and content.
    Cite the url of every result you rely on in your final answer.
    """
    response = tavily_client.search(
        query=question,
        max_results=5,
        include_answer=True,
    )
    results = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in response.get("results", [])
    ]

    # Show this step in the output so the retrieve -> evaluate -> search flow is visible
    print(f"[game_web_search] question={question!r}")
    for result in results:
        print(f"  - {result['title']} <{result['url']}>")

    if results:
        doc_id = cache_web_results(question, response.get("answer"), results)
        print(f"  -> saved to internal database as {doc_id}")

    return {"answer": response.get("answer"), "results": results}

In [7]:
# game_web_search saves its results to the internal DB, and that DB persists between runs.
# Without clearing, a re-run would answer "Mortal Kombat X" from the saved search, and the
# graded retrieve -> evaluate -> web search fallback would never show in the output.
# Set to False to keep earlier web searches and see the cache answer repeat questions.
CLEAR_WEB_CACHE_ON_START = True

if CLEAR_WEB_CACHE_ON_START:
    before = collection.count()
    clear_web_search_cache()
    print(f"Cleared {before - collection.count()} cached web search(es); {collection.count()} documents remain")

Cleared 1 cached web search(es); 15 documents remain


### Agent

In [8]:
# lib.agents.Agent runs on the course StateMachine:
# entry -> message_prep -> llm_processor <-> tool_executor -> termination

# Shared by every layer below (memory agent, state machine, JSON output) so they all count the same way
COUNT_RULE = (
    "If the question asks for a specific number of games, give exactly that many: no more and no "
    "fewer (fewer only if the information you have doesn't contain enough). Don't add extra "
    "\"also consider\" games beyond the number asked for."
)

UDAPLAY_INSTRUCTIONS = f"""
You are UdaPlay, an AI research assistant for the video game industry.

For every question, follow this process in order:
1. Call retrieve_game to search the internal game database. If the question refers to an earlier
   question in this conversation (e.g. "it", "that game"), rewrite it as a standalone question first.
2. Call evaluate_retrieval with the question and the documents retrieve_game returned.
3. If the evaluation says the documents are useful, answer from those documents.
   If it says they are not useful, call game_web_search and answer from the web results.
4. Base your answer only on what the tools returned. Never answer from memory alone.
   If the tools don't contain the answer, say you couldn't find it.
5. {COUNT_RULE}

Format your final answer exactly like this:
Answer: <a clear, concise answer to the question>
Source: <"UdaPlay internal database" or the URL(s) of the web results you used. If you used a "Cached web search" document from the database, also list the URLs in its Sources field>
Confidence: <high, medium or low. When answering from the database, use the confidence evaluate_retrieval gave. When answering from web search, rate how clearly and consistently the web results support your answer>
Process: <one sentence on which steps you took, e.g. "retrieved -> evaluated as not useful -> web search">
"""

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=UDAPLAY_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,  # deterministic tool choices so the process is consistent between runs
)

In [9]:
def ask(agent, query: str, session_id: str) -> None:
    """Run one query, printing the agent's steps, its final answer and token usage"""
    print(f"\n{'=' * 80}\nQuery: {query}\n{'=' * 80}")

    run = agent.invoke(query, session_id=session_id)

    final_state = run.get_final_state()
    if final_state is None:
        print("No final state produced")
        return

    print(f"\n{final_state['messages'][-1].content}")
    print(f"\n(total tokens: {final_state['total_tokens']})")


queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
]

# Separate session per query so each one shows its own retrieve -> evaluate -> search steps
for i, query in enumerate(queries, start=1):
    ask(udaplay_agent, query, session_id=f"query-{i}")


Query: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[retrieve_game] query='Pokémon Gold and Silver release date'
  - Pokémon Gold and Silver (Game Boy Color, 1999) distance=0.127
  - Pokémon Ruby and Sapphire (Game Boy Advance, 2002) distance=0.159
  - Mario Kart 8 Deluxe (Nintendo Switch, 2017) distance=0.233
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[evaluate_retrieval] useful=True confidence=high: The first retrieved document clearly states that Pokémon Gold and Silver was released in 1999. This directly answers the query about the release date of the game. The other documents do not pertain to the question, but since the first document provides the necessary information, I am confident that the documents are sufficient to respond to the query.
[StateMachine] Executing step: tool_executor
[StateMachine] Executing s

### Conversation state within a session

Two questions in the same session. The second one ("it") only makes sense if the agent remembers the first: the `Agent` keeps each session's messages in short-term memory and carries them into the next question.

In [10]:
session_id = "gta-session"

ask(udaplay_agent, "Who published Grand Theft Auto: San Andreas?", session_id=session_id)
# Follow-up that depends on the first question: "it" = Grand Theft Auto: San Andreas
ask(udaplay_agent, "What platform was it released on, and in what year?", session_id=session_id)

# Show the memory itself: both runs are stored in the session, and the second run
# started with the first run's conversation already loaded
session_runs = udaplay_agent.get_session_runs(session_id)
first_state = session_runs[0].get_final_state()
carried_over = len(first_state["messages"]) if first_state else 0
print(f"\nSession {session_id!r} holds {len(session_runs)} runs; "
      f"question 2 started with {carried_over} messages carried over from question 1")


Query: Who published Grand Theft Auto: San Andreas?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[retrieve_game] query='Who published Grand Theft Auto: San Andreas?'
  - Grand Theft Auto: San Andreas (PlayStation 2, 2004) distance=0.133
  - Gran Turismo (PlayStation 1, 1997) distance=0.212
  - Gran Turismo 5 (PlayStation 3, 2010) distance=0.216
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[evaluate_retrieval] useful=True confidence=high: The first retrieved document explicitly states that 'Grand Theft Auto: San Andreas' was published by 'Rockstar Games'. This directly answers the query about who published the game. The other documents do not pertain to the question, but they do not detract from the answer provided in the first document. Therefore, I am confident that the documents sufficiently answer the question.
[StateMachine] Executing step: tool_executor
[Sta

### (Optional) Advanced

#### Long-term memory

The user's preferences are stored as memory fragments in their own vector store. Before each query, the agent recalls the preferences relevant to it and adds them to its instructions:
- Prefer games on Sony systems when no platform is named
- Make answers interesting, not just a dry read of facts

In [11]:
USER_ID = "robert"
PREFERENCES_NAMESPACE = "preferences"

# Long-term memory lives in its own vector store, separate from the games collection
memory_db = VectorStoreManager(openai_api_key=os.environ["OPENAI_API_KEY"])
long_term_memory = LongTermMemory(memory_db)

long_term_memory.register(MemoryFragment(
    content=(
        "Prefers games on Sony systems (PlayStation 1 to 5, PSP, PS Vita) over any other platform. "
        "When the user doesn't name a platform and is choosing or asking for recommendations, pick only "
        "Sony PlayStation games whenever the results contain enough of them to answer. Use games from "
        "other platforms only to make up a shortfall, and don't mention non-Sony alternatives otherwise."
    ),
    owner=USER_ID,
    namespace=PREFERENCES_NAMESPACE,
))
long_term_memory.register(MemoryFragment(
    content=(
        "Prefers answers that aren't just a dry read of facts: include something interesting as well, "
        "such as a notable detail, a piece of trivia or some context about the game."
    ),
    owner=USER_ID,
    namespace=PREFERENCES_NAMESPACE,
))

# Tells the LLM that recalled preferences are rules it must follow, not suggestions
PREFERENCES_PREAMBLE = (
    "User preferences recalled from long-term memory. Treat them as rules, not suggestions, "
    "and apply them to your answer:"
)


class MemoryAgent:
    """UdaPlay agent with long-term memory: before each query it recalls the user's
    relevant preferences and adds them to the agent's instructions."""

    def __init__(self, agent: Agent, memory: LongTermMemory, owner: str, limit: int = 2):
        self.agent = agent
        self.memory = memory
        self.owner = owner
        self.limit = limit
        self.base_instructions = agent.instructions

    def invoke(self, query: str, session_id: Optional[str] = None) -> Run[AgentState]:
        recalled = self.memory.search(
            query_text=query,
            owner=self.owner,
            limit=self.limit,
            namespace=PREFERENCES_NAMESPACE,
        )

        # Show this step in the output so it's visible which preferences shaped the answer
        print("[long_term_memory] recalled preferences:")
        for fragment in recalled.fragments:
            print(f"  - {fragment.content}")

        preferences = "\n".join(f"- {fragment.content}" for fragment in recalled.fragments)
        self.agent.instructions = (
            f"{self.base_instructions}\n"
            f"{PREFERENCES_PREAMBLE}\n"
            f"{preferences}\n"
            "If a preference decides which game you pick, say so in the Answer. "
            "Keep any interesting detail accurate, ideally drawn from the tool results, "
            "and keep the Answer / Source / Confidence / Process format."
        )
        return self.agent.invoke(query, session_id=session_id)


# A separate Agent instance so the baseline udaplay_agent above stays memory-free
memory_agent = MemoryAgent(
    Agent(
        model_name="gpt-4o-mini",
        instructions=UDAPLAY_INSTRUCTIONS,
        tools=[retrieve_game, evaluate_retrieval, game_web_search],
        temperature=0.0,
    ),
    memory=long_term_memory,
    owner=USER_ID,
)

In [12]:
# Same 3 queries with long-term memory, plus a 4th proof of concept for the Sony preference:
# it names no platform, and the database has racing games on PlayStation 1 and 3 (Gran Turismo)
# as well as Nintendo Switch (Mario Kart 8 Deluxe), so the preference should steer it to Sony
memory_queries = queries + ["Can you recommend 2 good racing games?"]

for i, query in enumerate(memory_queries, start=1):
    ask(memory_agent, query, session_id=f"memory-query-{i}")


Query: When Pokémon Gold and Silver was released?
[long_term_memory] recalled preferences:
  - Prefers games on Sony systems (PlayStation 1 to 5, PSP, PS Vita) over any other platform. When the user doesn't name a platform and is choosing or asking for recommendations, pick only Sony PlayStation games whenever the results contain enough of them to answer. Use games from other platforms only to make up a shortfall, and don't mention non-Sony alternatives otherwise.
  - Prefers answers that aren't just a dry read of facts: include something interesting as well, such as a notable detail, a piece of trivia or some context about the game.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[retrieve_game] query='Pokémon Gold and Silver release date'
  - Pokémon Gold and Silver (Game Boy Color, 1999) distance=0.127
  - Pokémon Ruby and Sapphire (Game Boy Advance, 2002) distance=0.159
  - Mario Kart 8 Deluxe (Nintendo Sw

#### Agent as a state machine with tools as nodes

Instead of the LLM deciding which tool to call next, each tool is a fixed node and the workflow routes between them:

`recall_preferences -> retrieve_game -> evaluate_retrieval -> (useful? answer : game_web_search -> answer)`

The LLM is only used inside `evaluate_retrieval` (as judge) and to write the final answer.

In [13]:
from typing import Any, TypedDict
from lib.state_machine import StateMachine, Step, EntryPoint, Termination


class UdaPlayState(TypedDict):
    question: str
    preferences: list[str]  # recalled from long-term memory
    retrieved_docs: list[dict]  # from retrieve_game
    evaluation: dict  # from evaluate_retrieval
    web_results: dict  # from game_web_search (empty when the DB was enough)
    answer: str


answer_llm = LLM(model="gpt-4o-mini", temperature=0.0)


def recall_preferences_step(state: UdaPlayState) -> dict[str, Any]:
    recalled = long_term_memory.search(
        query_text=state["question"],
        owner=USER_ID,
        limit=2,
        namespace=PREFERENCES_NAMESPACE,
    )
    preferences = [fragment.content for fragment in recalled.fragments]
    print("[long_term_memory] recalled preferences:")
    for preference in preferences:
        print(f"  - {preference}")
    return {"preferences": preferences}


def retrieve_step(state: UdaPlayState) -> dict[str, Any]:
    return {"retrieved_docs": retrieve_game(state["question"])}


def evaluate_step(state: UdaPlayState) -> dict[str, Any]:
    return {"evaluation": evaluate_retrieval(state["question"], state["retrieved_docs"])}


def web_search_step(state: UdaPlayState) -> dict[str, Any]:
    return {"web_results": game_web_search(state["question"])}


def answer_step(state: UdaPlayState) -> dict[str, Any]:
    if state["web_results"]:
        context = json.dumps(state["web_results"], indent=2)
        source_rule = "the URL(s) of the web results you used"
        confidence_rule = "high, medium or low: how clearly and consistently the web results support your answer"
        process = "recall preferences -> retrieve -> evaluate (not useful) -> web search -> answer"
    else:
        context = json.dumps(state["retrieved_docs"], indent=2)
        source_rule = (
            '"UdaPlay internal database", plus the URLs in the Sources field of any '
            '"Cached web search" document you used'
        )
        # The judge already rated the documents the answer comes from
        confidence_rule = f'exactly "{state["evaluation"].get("confidence", "medium")}"'
        process = "recall preferences -> retrieve -> evaluate (useful) -> answer"

    preferences = "\n".join(f"- {preference}" for preference in state["preferences"])
    messages = [
        SystemMessage(content=(
            "You are UdaPlay, an AI research assistant for the video game industry. "
            "Answer the question using only the context provided. If it doesn't contain "
            "the answer, say you couldn't find it.\n"
            f"{COUNT_RULE}\n"
            f"{PREFERENCES_PREAMBLE}\n{preferences}\n"
            "If a preference decides which game you pick, say so. Keep any interesting "
            "detail accurate and drawn from the context.\n"
            "Format your answer exactly like this:\n"
            "Answer: <a clear, concise answer to the question>\n"
            f"Source: <{source_rule}>\n"
            f"Confidence: <{confidence_rule}>"
        )),
        UserMessage(content=f"Question: {state['question']}\n\nContext:\n{context}"),
    ]
    response = answer_llm.invoke(messages)

    # The workflow knows which path it took, so it reports the process itself
    return {"answer": f"{response.content}\nProcess: {process}"}


# Tools are fixed nodes; the only decision is the branch after evaluation
entry = EntryPoint[UdaPlayState]()
recall_node = Step[UdaPlayState]("recall_preferences", recall_preferences_step)
retrieve_node = Step[UdaPlayState]("retrieve_game", retrieve_step)
evaluate_node = Step[UdaPlayState]("evaluate_retrieval", evaluate_step)
web_search_node = Step[UdaPlayState]("game_web_search", web_search_step)
answer_node = Step[UdaPlayState]("answer", answer_step)
termination = Termination[UdaPlayState]()


def route_after_evaluation(state: UdaPlayState) -> Step[UdaPlayState]:
    return answer_node if state["evaluation"].get("useful") else web_search_node


udaplay_workflow = StateMachine[UdaPlayState](UdaPlayState)
udaplay_workflow.add_steps([
    entry, recall_node, retrieve_node, evaluate_node, web_search_node, answer_node, termination,
])
udaplay_workflow.connect(entry, recall_node)
udaplay_workflow.connect(recall_node, retrieve_node)
udaplay_workflow.connect(retrieve_node, evaluate_node)
udaplay_workflow.connect(evaluate_node, [answer_node, web_search_node], route_after_evaluation)
udaplay_workflow.connect(web_search_node, answer_node)
udaplay_workflow.connect(answer_node, termination)

In [14]:
for query in memory_queries:
    print(f"\n{'=' * 80}\nQuery: {query}\n{'=' * 80}")

    run = udaplay_workflow.run({
        "question": query,
        "preferences": [],
        "retrieved_docs": [],
        "evaluation": {},
        "web_results": {},
        "answer": "",
    })

    final_state = run.get_final_state()
    print(f"\n{final_state['answer'] if final_state else 'No final state produced'}")


Query: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[long_term_memory] recalled preferences:
  - Prefers games on Sony systems (PlayStation 1 to 5, PSP, PS Vita) over any other platform. When the user doesn't name a platform and is choosing or asking for recommendations, pick only Sony PlayStation games whenever the results contain enough of them to answer. Use games from other platforms only to make up a shortfall, and don't mention non-Sony alternatives otherwise.
  - Prefers answers that aren't just a dry read of facts: include something interesting as well, such as a notable detail, a piece of trivia or some context about the game.
[StateMachine] Executing step: recall_preferences
[retrieve_game] query='When Pokémon Gold and Silver was released?'
  - Pokémon Gold and Silver (Game Boy Color, 1999) distance=0.123
  - Pokémon Ruby and Sapphire (Game Boy Advance, 2002) distance=0.149
  - Super Mario 64 (Nintendo 64, 1996) distance=0.217
[StateMachine] 

#### Structured JSON output

Runs the same state-machine workflow, then has the LLM fill a `GameAnswer` schema (via OpenAI structured outputs) instead of writing free text. Every answer comes back as validated JSON: a list of games, each with `name`, `platform` and `release_date`, plus the answer's `confidence` (high, medium or low). The list holds exactly as many games as the question asks for (one by default).

In [15]:
class GameInfo(BaseModel):
    name: str = Field(description="Name of the game")
    platform: str = Field(description="Platform the game was released on, e.g. PlayStation 1, Nintendo 64")
    release_date: str = Field(description=(
        "Release date with only the precision the context gives: YYYY-MM-DD, YYYY-MM or YYYY. "
        "Never pad a year with -01-01"
    ))


class GameAnswer(BaseModel):
    games: list[GameInfo] = Field(description="The games that answer the question, in the order the answer gives them")
    confidence: Literal["high", "medium", "low"] = Field(description="The confidence given in the answer's Confidence line")


structured_llm = LLM(model="gpt-4o-mini", temperature=0.0)


def ask_structured(question: str) -> GameAnswer:
    """Run the state-machine workflow, then turn what it found into a GameAnswer JSON object"""
    run = udaplay_workflow.run({
        "question": question,
        "preferences": [],
        "retrieved_docs": [],
        "evaluation": {},
        "web_results": {},
        "answer": "",
    })
    final_state = run.get_final_state()
    if final_state is None:
        raise RuntimeError(f"Workflow produced no final state for: {question}")

    # Everything the workflow found, including its text answer (which already applied the
    # count rule and the user's preferences when choosing games, and states a confidence)
    context = {
        "answer": final_state["answer"],
        "retrieved_docs": final_state["retrieved_docs"],
        "web_results": final_state["web_results"],
    }
    messages = [
        SystemMessage(content=(
            "Extract the game(s) that answer the user's question, using only the context. "
            "Use exactly the games the answer gives, in the same order; if the question is about "
            "a single game, return one. "
            f"{COUNT_RULE} "
            "For release_date, copy only the precision the context gives (a year alone stays a "
            "year); if it isn't in the context, use \"Unknown\". "
            "Copy confidence from the answer's Confidence line."
        )),
        UserMessage(content=f"Question: {question}\n\nContext:\n{json.dumps(context, indent=2)}"),
    ]
    response = structured_llm.invoke(messages, response_format=GameAnswer)
    return PydanticOutputParser(model_class=GameAnswer).parse(response)


for query in memory_queries:
    print(f"\n{'=' * 80}\nQuery: {query}\n{'=' * 80}")
    answer = ask_structured(query)
    print(answer.model_dump_json(indent=2))


Query: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[long_term_memory] recalled preferences:
  - Prefers games on Sony systems (PlayStation 1 to 5, PSP, PS Vita) over any other platform. When the user doesn't name a platform and is choosing or asking for recommendations, pick only Sony PlayStation games whenever the results contain enough of them to answer. Use games from other platforms only to make up a shortfall, and don't mention non-Sony alternatives otherwise.
  - Prefers answers that aren't just a dry read of facts: include something interesting as well, such as a notable detail, a piece of trivia or some context about the game.
[StateMachine] Executing step: recall_preferences
[retrieve_game] query='When Pokémon Gold and Silver was released?'
  - Pokémon Gold and Silver (Game Boy Color, 1999) distance=0.123
  - Pokémon Ruby and Sapphire (Game Boy Advance, 2002) distance=0.149
  - Super Mario 64 (Nintendo 64, 1996) distance=0.217
[StateMachine] 